In [2]:
import pandas as pd
from typing import Any
import requests
import pandas as pd
import matplotlib.pyplot as plt
from time import sleep
import os
from dotenv import load_dotenv
import psycopg
import seaborn as sns
import numpy as np
from sklearn.ensemble import RandomForestRegressor 

In [ ]:
df=pd.read_pickle("ALL_10_YEARS")
columnas_enteras = [
    "altitud",
    "hrMedia",
    "hrMax",
    "hrMin",
    "dir" ,
    "horaPresMax",
    "horaPresMin" 
]
columnas_float =[
    "tmed",
    "prec",
    "tmin",
    "tmax",
    "pintMax",
    "velmedia",
    "racha",
    "presMax",
    "presMin",
    "sol"
]
columnas_hora=[
    "horatmin",
    "horatmax",
    "horaHrMax",
    "horaHrMin",
    "horaracha",
    
    "horaPIntMax"
]

columnas_horas_int=[
    "horaPresMax",
    "horaPresMin"
]



df["fecha"] = pd.to_datetime(df["fecha"],format="%Y-%m-%d")
df[columnas_enteras] = (df[columnas_enteras].apply(pd.to_numeric, errors="coerce").astype("Int64"))
df[columnas_float] = (df[columnas_float].apply(lambda columna: pd.to_numeric(columna.astype("string").str.strip().str.replace(",", ".", regex=False),errors="coerce")).astype("Float64"))

for columna in columnas_hora:
    df[f"{columna}_son_varias_h"] = (df[columna].eq("Varias"))
for columna in columnas_hora:
    df[columna] = (pd.to_datetime(df[columna].astype("string").str.strip(),format="%H:%M",errors="coerce").dt.strftime("%H:%M"))
    #.dt.strftime("%H:%M")
for columna in columnas_horas_int:
    df[columna] =df[columna].replace(24, 0).astype("Int64").astype("string") + ":00"
    df[columna] = (pd.to_datetime(df[columna].astype("string").str.strip(),format="%H:%M",errors="coerce").dt.strftime("%H:%M"))

df_limpio = df.astype(object).where(pd.notna(df), None)
df_limpio=df

In [79]:
import re

def normalizar_hora(valor):
    # Comprobamos si es nulo antes de convertir a string
    if pd.isna(valor):
        return np.nan
    valor = str(valor).strip()
    if valor in ("<NA>", "nan", "NaN", "None", ""):
        return np.nan
    if re.fullmatch(r"\d{1,2}", valor):
        return f"{int(valor):02d}:00"
    return valor

def convertir_datos_aemet(df):
    df = df.copy()

    # Fecha a datetime
    df["fecha"] = pd.to_datetime(df["fecha"], format="%Y-%m-%d", errors="coerce")

    # Categóricas / texto
    for col in ["indicativo", "provincia", "nombre"]:
        if col in df.columns:
            df[col] = df[col].astype("category")

    columnas_float = [
        "tmed", "prec", "tmin", "tmax", "hrMedia", "pintMax",
        "velmedia", "racha", "presMax", "presMin", "sol"
    ]

    columnas_hora = [
        "horatmin", "horatmax", "horaHrMax", "horaHrMin",
        "horaracha", "horaPresMax", "horaPresMin", "horaPIntMax"
    ]

    columnas_a_revisar = [c for c in columnas_float + columnas_hora if c in df.columns]

    # Detectar Acum / Varias / Ip
    mask_acum = pd.Series(False, index=df.index)
    mask_varias = pd.Series(False, index=df.index)
    mask_ip = pd.Series(False, index=df.index)

    for col in columnas_a_revisar:
        serie = df[col].apply(lambda x: str(x).strip().lower() if pd.notna(x) else np.nan)
        mask_acum |= (serie == "acum")
        mask_varias |= (serie == "varias")
        mask_ip |= (serie == "ip")

    df["precAcum"] = mask_acum.fillna(False)
    df["variasHoras"] = mask_varias.fillna(False)
    df["precIp"] = mask_ip.fillna(False)

    # Limpieza numérica
    def limpiar_numero(valor):
        if pd.isna(valor):
            return np.nan
        valor = str(valor).strip()
        if valor.lower() in ("ip",):
            return "0.05"
        if valor.lower() in ("acum", "varias", "<na>", "nan", "none", ""):
            return np.nan
        return valor.replace(",", ".")

    for col in columnas_float:
        if col in df.columns:
            df[col] = df[col].apply(limpiar_numero)
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # 4. Numéricos -> float
    for col in ["altitud", "hrMax", "hrMin", "dir"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype(float)

    # 5. Columnas de hora -> normalizadas a HH:MM
    for col in columnas_hora:
        if col in df.columns:
            df[col] = df[col].apply(normalizar_hora)

    return df

# df_datos_limpio = convertir_datos_aemet(df_datos)

# pd.set_option('display.max_columns', None)
# df_datos_limpio.sample(50)

In [80]:
df=pd.read_pickle("ALL_10_YEARS")
df=convertir_datos_aemet(df)


In [81]:
WINDOW_SIZE: int = 20



temp = df[df["indicativo"] == "7250C"].sort_values(by="fecha")[[
    "fecha",
    "tmed",
    "hrMedia"#,
    # "dia_sin",
    # "dia_cos"
    ]].reset_index(drop=True)
temp["dia_sin"] = np.sin(2*np.pi*temp["fecha"].dt.dayofyear/365)
temp["dia_cos"] = np.cos(2*np.pi*temp["fecha"].dt.dayofyear/365)

train = temp.iloc[10:-365]
test = temp.iloc[-365:]

rows = []
for i in range(train.shape[0]-WINDOW_SIZE):
    date=temp.iloc[i+WINDOW_SIZE]["fecha"]
    hrMedia=temp.iloc[i+WINDOW_SIZE]["hrMedia"]
    day_sin = temp.iloc[i+WINDOW_SIZE]["dia_sin"]
    day_cos = temp.iloc[i+WINDOW_SIZE]["dia_cos"]  
    days=temp["tmed"].loc[i:WINDOW_SIZE+i].to_numpy()

    final_row = np.hstack([
        date,
        day_sin,
        day_cos,
        hrMedia,
        days
    ])

    rows.append(final_row)


df_estacion=pd.DataFrame(rows)
df_estacion.iloc[:, 4] = pd.to_numeric(df_estacion.iloc[:, 4], errors="coerce")
df_estacion=df_estacion.dropna()
df_estacion_X : pd.DataFrame = df_estacion.iloc[:, 1:-1]
df_estacion_Y : pd.DataFrame = df_estacion.iloc[:, -1]
corte=365
df_estacion_X_train : pd.DataFrame =df_estacion_X.iloc[:-corte]
df_estacion_X_test : pd.DataFrame=df_estacion_X.iloc[-corte:]
df_estacion_Y_train : pd.DataFrame=df_estacion_Y.iloc[:-corte]
df_estacion_Y_test : pd.DataFrame=df_estacion_Y.iloc[-corte:]

In [82]:
df_estacion

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
0,2016-08-28,-0.845249,-0.534373,66.0,25.0,24.6,24.4,25.3,23.4,22.6,...,28.3,28.9,26.2,24.6,24.2,24.2,24.0,24.9,25.1,27.9
1,2016-08-29,-0.854322,-0.519744,70.0,24.6,24.4,25.3,23.4,22.6,24.5,...,28.9,26.2,24.6,24.2,24.2,24.0,24.9,25.1,27.9,26.8
2,2016-08-30,-0.863142,-0.504961,69.0,24.4,25.3,23.4,22.6,24.5,23.6,...,26.2,24.6,24.2,24.2,24.0,24.9,25.1,27.9,26.8,26.4
3,2016-08-31,-0.871706,-0.490029,69.0,25.3,23.4,22.6,24.5,23.6,26.1,...,24.6,24.2,24.2,24.0,24.9,25.1,27.9,26.8,26.4,25.0
4,2016-09-01,-0.880012,-0.474951,67.0,23.4,22.6,24.5,23.6,26.1,28.4,...,24.2,24.2,24.0,24.9,25.1,27.9,26.8,26.4,25.0,24.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3238,2025-07-10,-0.145799,-0.989314,67.0,28.0,28.0,28.8,27.4,27.6,27.4,...,29.0,28.6,28.3,28.2,27.8,28.6,28.3,29.4,28.2,27.6
3239,2025-07-11,-0.162807,-0.986658,68.0,28.0,28.8,27.4,27.6,27.4,29.6,...,28.6,28.3,28.2,27.8,28.6,28.3,29.4,28.2,27.6,27.8
3240,2025-07-12,-0.179767,-0.983709,44.0,28.8,27.4,27.6,27.4,29.6,27.5,...,28.3,28.2,27.8,28.6,28.3,29.4,28.2,27.6,27.8,29.1
3241,2025-07-13,-0.196673,-0.980469,57.0,27.4,27.6,27.4,29.6,27.5,28.3,...,28.2,27.8,28.6,28.3,29.4,28.2,27.6,27.8,29.1,27.4


In [83]:
df_estacion_X_train

,1,2,3,4,5,6,7,8,9,10,...,14,15,16,17,18,19,20,21,22,23
0,-0.845249,-0.534373,66.0,25.0,24.6,24.4,25.3,23.4,22.6,24.5,...,26.4,28.3,28.9,26.2,24.6,24.2,24.2,24.0,24.9,25.1
1,-0.854322,-0.519744,70.0,24.6,24.4,25.3,23.4,22.6,24.5,23.6,...,28.3,28.9,26.2,24.6,24.2,24.2,24.0,24.9,25.1,27.9
2,-0.863142,-0.504961,69.0,24.4,25.3,23.4,22.6,24.5,23.6,26.1,...,28.9,26.2,24.6,24.2,24.2,24.0,24.9,25.1,27.9,26.8
3,-0.871706,-0.490029,69.0,25.3,23.4,22.6,24.5,23.6,26.1,28.4,...,26.2,24.6,24.2,24.2,24.0,24.9,25.1,27.9,26.8,26.4
4,-0.880012,-0.474951,67.0,23.4,22.6,24.5,23.6,26.1,28.4,26.4,...,24.6,24.2,24.2,24.0,24.9,25.1,27.9,26.8,26.4,25.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2873,-0.162807,-0.986658,53.0,26.2,24.8,24.6,24.0,23.1,24.2,24.3,...,26.6,24.2,24.5,25.9,26.8,24.7,26.6,26.6,26.0,25.7
2874,-0.179767,-0.983709,60.0,24.8,24.6,24.0,23.1,24.2,24.3,26.0,...,24.2,24.5,25.9,26.8,24.7,26.6,26.6,26.0,25.7,29.0
2875,-0.196673,-0.980469,59.0,24.6,24.0,23.1,24.2,24.3,26.0,26.3,...,24.5,25.9,26.8,24.7,26.6,26.6,26.0,25.7,29.0,27.0
2876,-0.213521,-0.976938,67.0,24.0,23.1,24.2,24.3,26.0,26.3,27.0,...,25.9,26.8,24.7,26.6,26.6,26.0,25.7,29.0,27.0,29.2


In [90]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score

model: xgb.XGBRegressor = xgb.XGBRegressor(
    random_state=42,
    n_estimators=800,
    learning_rate=0.008
    )

model.fit(df_estacion_X_train, df_estacion_Y_train)
yhat: np.ndarray = model.predict(df_estacion_X_test)
print("MAE", mean_absolute_error(df_estacion_Y_test, yhat))
print("MSE", mean_squared_error(df_estacion_Y_test, yhat))
print("RMSE", root_mean_squared_error(df_estacion_Y_test, yhat))
print("R2", r2_score(df_estacion_Y_test, yhat))

MAE 1.2925420181065388
MSE 2.8925145232165885
RMSE 1.7007394048520745
R2 0.9268467164719186


In [24]:
modelo = RandomForestRegressor(
    n_estimators=1000,
    max_depth=10,
    random_state=42,
    min_samples_leaf=10,
    # criterion="poisson",
    bootstrap=True,
    n_jobs=-1
)

modelo.fit(df_estacion_X_train, df_estacion_Y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",1000
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"m

In [25]:
pred = modelo.predict(estacion_1_X_test)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("MAE :", mean_absolute_error(estacion_1_Y_test, pred))
print("RMSE:", np.sqrt(mean_squared_error(estacion_1_Y_test, pred)))
print("R²  :", r2_score(estacion_1_Y_test, pred))

MAE : 1.2898289119559625
RMSE: 1.7079333857785721
R²  : 0.926226543447013


In [52]:
def create_data_window(df:pd.DataFrame , date:str, indicativo:str, WINDOW :int = 10,) -> tuple[pd.DataFrame, float]:
    rows =[]
    df=df[df["indicativo"] == indicativo].sort_values(by="fecha")[["fecha","tmed","hrMedia","dia_sin","dia_cos"]].reset_index(drop=True)
    for i in range(train.shape[0]-WINDOW):
        dates=df.iloc[i+WINDOW]["fecha"]
        hrMedia=df.iloc[i+WINDOW]["hrMedia"]
        day_sin = df.iloc[i+WINDOW]["dia_sin"]
        day_cos = df.iloc[i+WINDOW]["dia_cos"]  
        days=df["tmed"].loc[i:WINDOW+i].to_numpy()

        final_row = np.hstack([
            dates,
            day_sin,
            day_cos,
            hrMedia,
            days
        ])
        rows.append(final_row)
        sdf=pd.DataFrame(rows)
        sdf = sdf[sdf.iloc[:, 0] == date]
        
    return (sdf.iloc[0, 1:-1], sdf.iloc[:, -1].iloc[0])

In [54]:
busca_b=create_data_window(df=df,date="2023-10-12",indicativo="7250C",WINDOW=20)
print(busca_b[0])
print(busca_b[1])

1    -0.981306
2     0.192452
3           73
4         23.2
5         21.2
6         21.5
7         21.2
8         23.0
9         23.0
10        22.6
11        22.9
12        23.4
13        24.2
14        23.0
15        23.0
16        23.8
17        22.8
18        22.6
19        21.6
20        22.4
21        21.5
22        21.9
23        22.1
Name: 2601, dtype: object
21.7


In [57]:
valor_a_predecir=create_data_window(
        df=df,
        date="2023-12-31",
        indicativo="0034X",
        WINDOW=20)
prediccion_from_modelos = modelos["0034X"].predict((valor_a_predecir[0].to_numpy().reshape(1, -1)))
print(prediccion_from_modelos[0])
print(valor_a_predecir[1])

9.258559740898306
8.2


In [37]:
modelos

{'0009X': RandomForestRegressor(max_depth=10, min_samples_leaf=10, n_estimators=1000,
                       n_jobs=-1, random_state=42),
 '0016A': RandomForestRegressor(max_depth=10, min_samples_leaf=10, n_estimators=1000,
                       n_jobs=-1, random_state=42),
 '0016B': RandomForestRegressor(max_depth=10, min_samples_leaf=10, n_estimators=1000,
                       n_jobs=-1, random_state=42),
 '0034X': RandomForestRegressor(max_depth=10, min_samples_leaf=10, n_estimators=1000,
                       n_jobs=-1, random_state=42),
 '0042Y': RandomForestRegressor(max_depth=10, min_samples_leaf=10, n_estimators=1000,
                       n_jobs=-1, random_state=42),
 '0061X': RandomForestRegressor(max_depth=10, min_samples_leaf=10, n_estimators=1000,
                       n_jobs=-1, random_state=42),
 '0066X': RandomForestRegressor(max_depth=10, min_samples_leaf=10, n_estimators=1000,
                       n_jobs=-1, random_state=42),
 '0073X': RandomForestRegressor(ma

In [28]:
df["indicativo"].nunique()

894

In [36]:
modelos = {}
estaciones_no_modelas=[]
cont=0
for indicativo, df_estacion in df.groupby("indicativo"):

    # df_estacion = df_estacion.dropna(subset=features + [target])

    WINDOW_SIZE: int = 20

    df_estacion["dia_sin"] = np.sin(2*np.pi*df_estacion["fecha"].dt.dayofyear/365)
    df_estacion["dia_cos"] = np.cos(2*np.pi*df_estacion["fecha"].dt.dayofyear/365)

    temp = df_estacion.sort_values(by="fecha")[[
        "fecha",
        "tmed",
        "hrMedia",
        "dia_sin",
        "dia_cos"
        ]].reset_index(drop=True)


    train = temp.iloc[10:-365]
    test = temp.iloc[-365:]

    rows = []
    for i in range(train.shape[0]-WINDOW_SIZE):
        date=temp.iloc[i+WINDOW_SIZE]["fecha"]
        hrMedia=temp.iloc[i+WINDOW_SIZE]["hrMedia"]
        day_sin = temp.iloc[i+WINDOW_SIZE]["dia_sin"]
        day_cos = temp.iloc[i+WINDOW_SIZE]["dia_cos"]  
        days=temp["tmed"].loc[i:WINDOW_SIZE+i].to_numpy()

        final_row = np.hstack([
            date,
            day_sin,
            day_cos,
            hrMedia,
            days
        ])

        rows.append(final_row)


    df_estacion=pd.DataFrame(rows)
    df_estacion=df_estacion.dropna()
    if len(df_estacion) < 1500:
        estaciones_no_modelas.append(indicativo)
        cont=1+cont
        print(f"La estación {indicativo} no tiene suficientes datos. Total de estaciones no modeladas {len(estaciones_no_modelas)}")
        continue

    df_estacion_X : pd.DataFrame = df_estacion.iloc[:, 1:-1]
    df_estacion_Y : pd.DataFrame = df_estacion.iloc[:, -1]
    corte=365
    df_estacion_X_train : pd.DataFrame =df_estacion_X.iloc[:-corte]
    df_estacion_X_test : pd.DataFrame=df_estacion_X.iloc[-corte:]
    df_estacion_Y_train : pd.DataFrame=df_estacion_Y.iloc[:-corte]
    df_estacion_Y_test : pd.DataFrame=df_estacion_Y.iloc[-corte:]

   

    X = df_estacion_X_train

    y = df_estacion_Y_train

    modelo = RandomForestRegressor(
        n_estimators=1000,
        max_depth=10,
        random_state=42,
        min_samples_leaf=10,
        # criterion="poisson",
        bootstrap=True,
        n_jobs=-1
    )

    modelo.fit(X, y)
    cont=1+cont
    print(f"Modelo de estación {indicativo} entrenado. Total: {cont} de {df["indicativo"].nunique()}. Total de estaciones no modeladas {len(estaciones_no_modelas)}")
    modelos[indicativo] = modelo

La estación 0002I no tiene suficientes datos. Total de estaciones 1
modelo de estación 0009X entrenado. Total: 2 de 894.
modelo de estación 0016A entrenado. Total: 3 de 894.
modelo de estación 0016B entrenado. Total: 4 de 894.
modelo de estación 0034X entrenado. Total: 5 de 894.
modelo de estación 0042Y entrenado. Total: 6 de 894.
modelo de estación 0061X entrenado. Total: 7 de 894.
modelo de estación 0066X entrenado. Total: 8 de 894.
modelo de estación 0073X entrenado. Total: 9 de 894.
modelo de estación 0076 entrenado. Total: 10 de 894.
modelo de estación 0092X entrenado. Total: 11 de 894.
modelo de estación 0106X entrenado. Total: 12 de 894.
modelo de estación 0114X entrenado. Total: 13 de 894.
modelo de estación 0120X entrenado. Total: 14 de 894.
modelo de estación 0149D entrenado. Total: 15 de 894.
modelo de estación 0149X entrenado. Total: 16 de 894.
modelo de estación 0158O entrenado. Total: 17 de 894.
modelo de estación 0158X entrenado. Total: 18 de 894.
modelo de estación 0171